#### SummarizationMiddleware中间价

In [7]:
from re import finditer

from sqlalchemy import false

"""
举例1
"""
from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware, PIIMiddleware
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

from llm.my_llm import model
from langchain.agents import create_agent


agent=create_agent(
    model=model,
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=[
                ("tokens", 300),
                ("messages", 5),
                ("fraction", 0.5)
            ],
            keep=('messages',2)
        )
    ]
)
messages=[
    SystemMessage("你是个非常友好的AI助手"),
    HumanMessage("你好啊，我是老王，你是谁"),
    AIMessage("你好老王，我是小王"),
    HumanMessage('好的，小王，很高兴认识您'),
    AIMessage('不要高兴的太早'),
    HumanMessage('哈哈哈，什么意思')
]
res=agent.invoke({
    "messages":messages
})

for msg in res['messages']:
    msg.pretty_print()




================================ Human Message =================================

Here is a summary of the conversation to date:

## SESSION INTENT
建立初步联系与身份确认；用户（老王）与AI助手（小王）进行友好问候，为后续交互奠定基础。

## SUMMARY
- 系统预设AI角色为非常友好的助手。
- 用户自我介绍为“老王”，并询问AI身份。
- AI以“小王”身份回应，保持友好语气完成互认。
- 用户表达结识意愿（“很高兴认识您”）。
- 当前对话仅处于破冰/寒暄阶段，未产生具体任务、技术决策、策略选择或已验证/排除的选项。

## ARTIFACTS
None

## NEXT STEPS
等待用户提出明确的需求、问题或任务目标，以便开始实质性工作。
================================== Ai Message ==================================

不要高兴的太早
================================ Human Message =================================

哈哈哈，什么意思
================================== Ai Message ==================================

哈哈，老王您别多想～我其实就是句玩笑话，意思是：“咱们刚互相认识，正事儿还没开始呢，您可别提前给我‘加戏’啦！” 😄

说正经的，作为您的专属助手小王，我已经准备好随时接单了。接下来您是想随便聊聊，还是有具体的问题、任务或想法需要我帮忙？您一句话，我马上跟上！


In [5]:
f"""
    举例2：使用summary_prompt,摘要时自定义提示词：该提示词需要包含{messages}
"""
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

from llm.my_llm import model
from langchain.agents import create_agent


agent=create_agent(
    model=model,
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=[
                ("tokens", 300),
                ("messages", 5),
                ("fraction", 0.5)
            ],
            keep=('messages',2),
            summary_prompt="对历史消息摘要，消息列表如下\n{messages}"
        )
    ]
)
messages=[
    SystemMessage("你是个非常友好的AI助手"),
    HumanMessage("你好啊，我是老王，你是谁"),
    AIMessage("你好老王，我是小王"),
    HumanMessage('好的，小王，很高兴认识您'),
    AIMessage('不要高兴的太早'),
    HumanMessage('哈哈哈，什么意思')
]
res=agent.invoke({
    "messages":messages
})

for msg in res['messages']:
    msg.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

这段历史消息摘要如下：

**对话阶段**：初次问候与自我介绍  
**核心内容**：用户（老王）主动打招呼并询问AI身份，AI（小王）以友好态度完成自我说明；用户随即表达结识意愿，双方完成基础身份确认。  
**当前状态**：关系建立初期，沟通氛围轻松礼貌，已具备进一步开展具体需求对话的基础。
================================== Ai Message ==================================

不要高兴的太早
================================ Human Message =================================

哈哈哈，什么意思
================================== Ai Message ==================================

哈哈，被你这么一问我还真得认真琢磨一下～

“别高兴得太早”在咱们的语境里，通常有两层意思：
1. **现实向**：提醒接下来的任务可能有难度、有坑，或者需求比较“硬核”，让我别提前拍胸脯保证；
2. **玩笑向**：朋友间聊天时故意“泼点冷水”调节气氛，属于熟人间的调侃。

结合你前面那句“哈哈哈”，我猜大概率是第二种，顺便给我打个预防针。放心，我这人（AI）抗造得很，不怕复杂也不怕绕。你尽管把真实想法或需求抛过来，是写代码、查资料、理思路，还是单纯想测测我的底线，我都接着。

毕竟“老王”都开口了，“小王”哪敢松懈呀～ 你说吧，今天想搞点啥？


### HumanInTheLoopMiddleware（人工审核）

In [ ]:
"""
    举例1
"""
from langchain_core.messages import HumanMessage
from langchain.tools import tool
from langchain.agents import create_agent
from llm.my_llm import model_tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware
from rich import print as rprint
from langgraph.types import Command
@tool
def get_weather(city:str,is_forcast:bool=False)->str:
    """
        查询指定城市天气

        Args:
            city : 城市名称
            is_forcast : 是否查询预报
    """
    res=f'{city}今天天气不错'

    if is_forcast:
        res+='\n明天下雨'

    return res

@tool
def get_news()->str:
    """
        查询当然新闻
    """
    return '中国又赢了'

@tool
def get_day()->str:
    """获取今天是星期几"""
    return "今天是星期四"

@tool
def send_email()->str:
    """发送邮件"""
    return '发送邮件成功'


agent=create_agent(
    model=model_tool,
    tools=[get_weather,get_news,get_day,send_email],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(interrupt_on={
            "get_weather":True,
            "get_news":False,
            "send_email":True,
            "get_day":{
                    "allowed_decisions":["approve", "reject"],
                    "description":"查询日期中断"
            },
        },
         description_prefix='中断啦'
        ),
    ]
)
config = {"configurable": {"thread_id": "conversation-001"}}
res=agent.invoke({
    "messages":[
        HumanMessage("今天是星期几？今天的新闻告诉我？北京今天天气也告诉我？邮件发送成功了？")
    ]
},config=config)
rprint(res)

In [22]:


"""
    举例2
"""
weather_decision={
    "type":"edit",
    "edited_action":{
        "name":"get_weather",
        "args":{"city":"重庆市","is_forcast":True}
    }
}
day_decision={
    "type":"approve"
}
send_email_decision={
    "type":"approve"
}
decisions={
    "decisions":[]
}
interrupt=res.get('__interrupt__',[])
action_requests=interrupt[0].value["action_requests"]

for action in action_requests:
    if action['name']=='get_day':
        decisions['decisions'].append(day_decision)
    if action['name']=='get_weather':
        decisions['decisions'].append(weather_decision)
    if action['name']=='send_email':
        decisions['decisions'].append(send_email_decision)

if interrupt:
    """Command引入到是from langgraph.types import Command中的"""
    resume_res=agent.invoke(
        Command(resume=decisions),
        config=config
    )
    for re in resume_res['messages']:
        re.pretty_print()


================================ Human Message =================================

今天是星期几？今天的新闻告诉我？北京今天天气也告诉我？邮件发送成功了？
================================== Ai Message ==================================
Tool Calls:
  get_day (call_7e4079a33d7849feb944394e)
 Call ID: call_7e4079a33d7849feb944394e
  Args:
  get_news (call_5895aca3a6ea4bbea1f476c6)
 Call ID: call_5895aca3a6ea4bbea1f476c6
  Args:
  get_weather (call_2972fae8eaee46a7bc7e7323)
 Call ID: call_2972fae8eaee46a7bc7e7323
  Args:
    city: 重庆市
    is_forcast: True
  send_email (call_5034551a096346d9b5b7feb5)
 Call ID: call_5034551a096346d9b5b7feb5
  Args:
================================= Tool Message =================================
Name: get_day

今天是星期四
================================= Tool Message =================================
Name: get_news

中国又赢了
================================= Tool Message =================================
Name: get_weather

重庆市今天天气不错
明天下雨
================================= Tool Message ==================

### PIIMiddleware中间价

In [16]:
# 举例1 ： 使用内置检测器
from langchain.agents import create_agent
from llm.my_llm import model_tool
from langchain.agents.middleware.pii import PIIMiddleware
from langchain_core.messages import HumanMessage
from rich import print as rprint
agent=create_agent(
    model=model_tool,
    middleware=[
        PIIMiddleware("email", strategy="redact"),
        PIIMiddleware("credit_card", strategy="mask"),
        PIIMiddleware("ip", strategy="block"),
        PIIMiddleware("mac_address", strategy="mask"),
        PIIMiddleware("url", strategy="hash"),
    ]
)
"""
    redact:用“[REDACTED_TYPE]'占位符替换个人身份信息
    block:检测到个人身份信息时提出异常（During task with name 'PIIMiddleware[ip].before_model' and id 'a06e5781-18b1-3b99-d4a9-3b9a4bf91fe9'）直接报错
    mask:部分掩盖个人身份信息（PII）（例如，信用卡的“*******1234”）
    hash:用确定性哈希替换PII（例如，'<email_hash：a1b2c3d4>'）
"""
# "    IP地址是：192.168.1.12",
res=agent.invoke({
    "messages":[
        HumanMessage(""""
    "    帮我向 156168188@qq.com 发送一封邮件\n",
    "    同时查看银行卡号： 5105-1051-0510-5100 的余额\n",
    "    访问 https://localhost:12345\n",

    "    确认这是不是 MAC地址： 11-11-11-11-11-11\n""")
    ]
})
# rprint(res)
# print(res['messages'])
for r in res['messages']:
    r.pretty_print()


================================ Human Message =================================

"
    "    帮我向 [REDACTED_EMAIL] 发送一封邮件
",
    "    同时查看银行卡号： ****-****-****-5100 的余额
",
    "    访问 <url_hash:dd5fc2a9>
",

    "    确认这是不是 MAC地址： **-**-**-**-**-11

================================== Ai Message ==================================

您好！针对您列出的四项需求，我逐一说明我的能力边界并提供可操作的替代方案：

1. **📧 发送邮件**  
   我无法直接代为发送电子邮件，但可以：
   - 为您起草完整邮件（含主题、正文、附件提示等）
   - 提供主流邮箱（Outlook/Gmail/企业微信/钉钉等）的发信步骤截图指引
   - 检查邮件语法、语气或合规性  
   👉 请提供收件人、主题及核心内容，我将立即为您生成可直接复制发送的模板。

2. **🏦 查看银行卡余额**  
   **我无法访问任何银行系统、数据库或金融平台，也不能处理账户余额查询请求。**  
   🔒 出于资金安全与隐私保护原则，此类操作请务必通过以下官方渠道完成：
   - 银行官方手机APP / 网上银行
   - 拨打卡片背面客服电话
   - 前往网点柜台或智能柜员机  
   ⚠️ 请勿在任何非官方渠道输入完整卡号、密码、短信验证码或身份证信息。

3. **🔗 访问 `<url_hash:dd5fc2a9>`**  
   - 该格式并非标准网址（缺少 `https://` 协议头与域名/路径）
   - 我目前不具备实时打开网页、解析哈希链接或执行外部跳转的功能  
   ✅ 建议：若您有完整网址，可直接在浏览器中粘贴访问；若为内部系统/短链/加密链接，请联系对应平台的技术支持获取有效地址。

4. **🌐 确认是否为 MAC 地址：`**-**-**-**-**-11`**  
   - 标准 MAC 地址由 **6 组十六进制字符** 组成，分隔符

In [18]:
# 使用IP的block监测
try:
    res=agent.invoke({
        "messages":[HumanMessage("IP地址是：192.168.1.12")]
    })
except Exception as e:
    print(f'监测到IP，抛出异常{e}')



监测到IP，抛出异常Detected 1 instance(s) of ip in text content


In [23]:
# 举例2:自定义检测器/函数
import re

# 自定义检测函数
def detect_phone_number(content:str):
    return [
        {
            "text":m.group(0), # 提取到具体匹配到的11位数字文本
            "start":m.start(), # 这段数字在原文本中的“起始索引位置”（从0开始算）
            "end":m.end()      # 这段数字在原文本中“结束索引位置”
        } for m in re.finditer(r'[0-9]{11}',content)
    ]




In [24]:
text='大家好，我的电话是13996615551'
result=detect_phone_number(text)
print(result)

[{'text': '13996615551', 'start': 9, 'end': 20}]


In [28]:
from langchain.agents import create_agent
from llm.my_llm import model_tool
from langchain.agents.middleware.pii import PIIMiddleware
from langchain_core.messages import HumanMessage
from rich import print as rprint
agent=create_agent(
    model=model_tool,
    middleware=[
        PIIMiddleware('phone_number',strategy='mask',detector=detect_phone_number)
    ]
)
response=agent.invoke({
    "messages":[HumanMessage("帮我给这个号码打电话：13996615551")]
})
for msg in response['messages']:
    msg.pretty_print()

================================ Human Message =================================

帮我给这个号码打电话：****5551
================================== Ai Message ==================================

我无法直接为您拨打电话，因为作为人工智能助手，我没有接入电话网络或执行拨号操作的功能。此外，您提供的号码 `****5551` 是部分隐藏的，缺少前缀区号及中间位数，即使有拨号能力也无法完整呼出。

💡 **建议您可以：**
1. 在手机**通话记录、短信或通讯录**中搜索该号码的后四位，找到完整信息后手动拨打；
2. 如果知道对方使用的社交软件（微信、钉钉、企业微信等），可直接发送文字或语音留言；
3. 若是工作/业务号码，可通过公司官网、客服渠道或相关平台查询公开联系方式；
4. 如需我帮您**起草一段礼貌的留言/短信内容**，请告诉我具体事由和对方身份，我会立即为您生成。

随时告诉我您需要哪方面的协助！
